# The Pace of Life

**Nguồn dữ liệu:** R.V. Levine, "The Pace of Life", *American Scientist* 78 (1990): 450–59.

## Bối cảnh
Một giả thuyết phổ biến trong tâm lý học cho rằng những người luôn có cảm giác gấp gáp về thời gian
(hành vi *type-A*) dễ mắc bệnh tim hơn những người sống thong thả (*type-B*). Để kiểm định giả thuyết này
ở cấp độ thành phố, các nhà nghiên cứu đã chọn 36 thành phố của Mỹ (mỗi vùng miền chọn 3 đô thị lớn, 3 đô
thị vừa, 3 đô thị nhỏ) và đo 3 chỉ số phản ánh nhịp sống:

- **Walk** — tốc độ đi bộ của người đi đường trên quãng đường 20 mét trong giờ hành chính (chỉ số càng cao ⇒ đi càng nhanh ⇒ nhịp sống càng gấp gáp).
- **Bank** — thời gian trung bình một nhân viên ngân hàng trả tiền thối cho 2 tờ 20 đô-la (chỉ số càng cao ⇒ xử lý càng chậm ⇒ nhịp sống càng thong thả, ngược hướng với Walk/Talk).
- **Talk** — tốc độ nói của nhân viên bưu điện, đo bằng số âm tiết chia cho thời gian trả lời (chỉ số càng cao ⇒ nói càng nhanh ⇒ nhịp sống càng gấp gáp).

Đồng thời, nhóm nghiên cứu thu thập **Heart** — tỷ lệ tử vong do bệnh tim đã hiệu chỉnh theo tuổi của mỗi
thành phố.

## Câu hỏi nghiên cứu
> Dữ liệu có ủng hộ hay bác bỏ niềm tin rằng những người có cảm giác gấp gáp về thời gian dễ mắc bệnh
> tim hơn không?

## Các bước nghiên cứu phân tích
1. Thống kê mô tả và kiểm tra chất lượng dữ liệu.
2. Phân tích tương quan Pearson giữa từng chỉ số nhịp sống và Heart.
3. Trực quan hoá: scatterplot, ma trận tương quan, pairplot.
4. Hồi quy tuyến tính bội `Heart ~ Walk + Bank + Talk` và kiểm tra các giả định hồi quy (đa cộng
   tuyến, phân phối phần dư, phương sai đồng nhất, điểm ảnh hưởng).
5. Xây dựng chỉ số nhịp sống tổng hợp (composite pace-of-life index) và hồi quy đơn biến với Heart để
   có một phép kiểm định tổng hợp, dễ diễn giải.
6. Ước lượng hồi quy tuyến tính Bayes cho cùng mô hình `Heart ~ Walk + Bank + Talk`: đặt prior liên
   hợp Normal-Inverse-Gamma, suy ra hậu nghiệm dạng đóng, lấy mẫu hậu nghiệm bằng Gibbs sampling (MCMC),
   chẩn đoán hội tụ (trace plot, Gelman-Rubin R-hat) và đối chiếu với ước lượng tần suất (OLS).
7. Đánh giá kết quả (tần suất + Bayes), trả lời câu hỏi nghiên cứu và nêu các giới hạn của phân tích.


In [6]:
import pandas as pd
import numpy as np
import scipy.stats as stats
import statsmodels.api as sm
from statsmodels.stats.outliers_influence import variance_inflation_factor
from statsmodels.stats.stattools import jarque_bera
import matplotlib.pyplot as plt
import seaborn as sns
import os

plt.rcParams['figure.figsize'] = (8, 5)
plt.rcParams['axes.grid'] = True
sns.set_theme(style='whitegrid', palette='deep')

FIG_DIR = '../figures'
os.makedirs(FIG_DIR, exist_ok=True)

pd.set_option('display.precision', 3)


Exception ignored in PyObject_HasAttr(); consider using PyObject_HasAttrWithError(), PyObject_GetOptionalAttr() or PyObject_GetAttr():
Traceback (most recent call last):
  File "<frozen importlib._bootstrap>", line 491, in _call_with_frames_removed
AttributeError: partially initialized module 'pandas' from 'C:\Users\SONTUNG\AppData\Roaming\Python\Python314\site-packages\pandas\__init__.py' has no attribute '_pandas_parser_CAPI' (most likely due to a circular import)


## 1. Nạp và kiểm tra dữ liệu

In [ ]:
df = pd.read_csv('../data/pace_of_life.csv')
print(f"So thanh pho: {df.shape[0]}, so bien: {df.shape[1]}")
df.head(10)

In [ ]:
df.info()
print()
print("So gia tri thieu theo cot:")
print(df.isna().sum())

**Nhận xét:** Dữ liệu có 36 thành phố, đúng như mô tả gốc của Levine, không có giá trị khuyết. Cả 4 biến
định lượng đều đã được đưa về thang đo không đơn vị (theo ghi chú gốc của bộ dữ liệu).

## 2. Thống kê mô tả

In [ ]:
desc = df[['Walk', 'Bank', 'Talk', 'Heart']].describe().T
desc['skew'] = df[['Walk', 'Bank', 'Talk', 'Heart']].skew()
desc

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(16, 4))
for ax, col in zip(axes, ['Walk', 'Bank', 'Talk', 'Heart']):
    sns.boxplot(y=df[col], ax=ax, color='#4C72B0')
    sns.stripplot(y=df[col], ax=ax, color='black', alpha=0.5, size=4, jitter=0.15)
    ax.set_title(col)
plt.suptitle('Phan phoi cac bien (boxplot + diem du lieu)')
plt.tight_layout()
plt.savefig(f'{FIG_DIR}/01_boxplots.png', dpi=150)
plt.show()

## 3. Chỉ số nhịp sống tổng hợp (Pace-of-Life Index)

Vì `Bank` đo **thời gian xử lý** (càng cao càng chậm) trong khi `Walk` và `Talk` đo **tốc độ** (càng cao
càng nhanh), ta cần đảo dấu `Bank` trước khi kết hợp ba chỉ số thành một thước đo "nhịp sống gấp gáp"
(time urgency) duy nhất. Mỗi biến được chuẩn hoá (z-score) trước khi cộng để tránh một biến có phương sai
lớn hơn lấn át các biến khác.

In [ ]:
z = df[['Walk', 'Bank', 'Talk']].apply(stats.zscore)
df['Pace_Index'] = (z['Walk'] + (-z['Bank']) + z['Talk']) / 3
df[['City', 'Walk', 'Bank', 'Talk', 'Pace_Index', 'Heart']].sort_values('Pace_Index', ascending=False).head(5)

## 4. Phân tích tương quan

In [ ]:
vars_ = ['Walk', 'Bank', 'Talk', 'Pace_Index', 'Heart']
corr = df[vars_].corr(method='pearson')
corr

In [ ]:
print("Tuong quan Pearson voi Heart (kem kiem dinh y nghia thong ke, alpha = 0.05)")
print("-" * 65)
for col in ['Walk', 'Bank', 'Talk', 'Pace_Index']:
    r, p = stats.pearsonr(df[col], df['Heart'])
    sig = 'co y nghia' if p < 0.05 else 'khong co y nghia'
    direction = 'thuan (+)' if r > 0 else 'nghich (-)'
    print(f"{col:12s} r = {r:+.3f}   p = {p:.4f}   -> tuong quan {direction}, {sig}")

In [ ]:
fig, ax = plt.subplots(figsize=(6, 5))
mask = np.triu(np.ones_like(corr, dtype=bool), k=1)
sns.heatmap(corr, mask=~np.tril(np.ones_like(corr, dtype=bool)) & mask, annot=True, fmt='.2f',
            cmap='RdBu_r', vmin=-1, vmax=1, square=True, cbar_kws={'label': 'He so tuong quan Pearson'})
plt.title('Ma tran tuong quan')
plt.tight_layout()
plt.savefig(f'{FIG_DIR}/02_correlation_heatmap.png', dpi=150)
plt.show()

**Diễn giải chiều tương quan (lưu ý `Bank` đo thời gian xử lý, không phải tốc độ; giả thuyết type-A dự
đoán Heart tăng khi Walk tăng, Talk tăng và Bank GIẢM):**

- `Walk`: r = +0.348, p = 0.038 (**có ý nghĩa**) — đúng chiều giả thuyết.
- `Talk`: r = +0.100, p = 0.562 (không có ý nghĩa) — đúng chiều nhưng quá yếu.
- `Bank`: r = +0.318, p = 0.059 (cận biên) — **ngược** chiều giả thuyết: xử lý giao dịch càng chậm (nhịp
  sống càng thong thả) lại đi kèm Heart càng cao. Vì vậy khi đảo dấu Bank để đưa vào `Pace_Index`, thành
  phần này kéo chỉ số tổng hợp về hướng ngược với Walk và Talk.
- `Pace_Index`: r = +0.077, p = 0.654 (không có ý nghĩa) — hệ quả của việc Walk/Talk và Bank triệt tiêu
  lẫn nhau.

## 5. Scatterplot: Heart theo từng chỉ số nhịp sống

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))
for ax, col in zip(axes, ['Walk', 'Bank', 'Talk']):
    sns.regplot(data=df, x=col, y='Heart', ax=ax, scatter_kws={'alpha': 0.7}, line_kws={'color': 'red'})
    r, p = stats.pearsonr(df[col], df['Heart'])
    ax.set_title(f'Heart vs {col}\nr = {r:.2f}, p = {p:.3f}')
plt.tight_layout()
plt.savefig(f'{FIG_DIR}/03_scatter_predictors.png', dpi=150)
plt.show()


In [ ]:
fig, ax = plt.subplots(figsize=(6.5, 5))
sns.regplot(data=df, x='Pace_Index', y='Heart', ax=ax, scatter_kws={'alpha': 0.7}, line_kws={'color': 'red'})
r, p = stats.pearsonr(df['Pace_Index'], df['Heart'])
ax.set_title(f'Heart vs Pace_Index (chi so nhip song gop)\nr = {r:.2f}, p = {p:.4f}')
for _, row in df.iterrows():
    if abs(row['Pace_Index']) > 1.3 or row['Heart'] > 29 or row['Heart'] < 12:
        ax.annotate(row['City'], (row['Pace_Index'], row['Heart']), fontsize=8, alpha=0.7,
                    xytext=(4, 4), textcoords='offset points')
plt.tight_layout()
plt.savefig(f'{FIG_DIR}/04_scatter_pace_index.png', dpi=150)
plt.show()


In [ ]:
g = sns.pairplot(df[['Walk', 'Bank', 'Talk', 'Heart']], corner=True, diag_kind='kde',
                  plot_kws={'alpha': 0.6})
g.fig.suptitle('Pairplot cac bien', y=1.02)
g.savefig(f'{FIG_DIR}/05_pairplot.png', dpi=150)
plt.show()


## 6. Hồi quy tuyến tính bội: `Heart ~ Walk + Bank + Talk`

In [ ]:
X = df[['Walk', 'Bank', 'Talk']]
X = sm.add_constant(X)
y = df['Heart']

model = sm.OLS(y, X).fit()
print(model.summary())


**Cách đọc kết quả:**
- `R-squared` / `Adj. R-squared`: tỷ lệ phương sai của `Heart` được giải thích bởi 3 biến nhịp sống.
- `F-statistic` và `Prob (F-statistic)`: kiểm định ý nghĩa tổng thể của mô hình (H0: tất cả hệ số hồi
  quy = 0).
- Với mỗi biến, cột `coef` là hệ số hồi quy riêng phần (kiểm soát 2 biến còn lại), `P>|t|` là p-value
  kiểm định `H0: beta = 0`.

### 6.1. Kiểm tra đa cộng tuyến (VIF)

In [ ]:
vif = pd.DataFrame()
vif['Variable'] = X.columns
vif['VIF'] = [variance_inflation_factor(X.values, i) for i in range(X.shape[1])]
vif


VIF < 5 cho các biến độc lập cho thấy đa cộng tuyến không phải là vấn đề nghiêm trọng
trong mô hình này (nếu kết quả thực tế khác, cần xem lại khi chạy notebook).

### 6.2. Chẩn đoán phần dư (residual diagnostics)

In [ ]:
fitted = model.fittedvalues
resid = model.resid
influence = model.get_influence()
standardized_resid = influence.resid_studentized_internal
cooks_d = influence.cooks_distance[0]

fig, axes = plt.subplots(2, 2, figsize=(12, 9))

axes[0, 0].scatter(fitted, resid, alpha=0.7)
axes[0, 0].axhline(0, color='red', linestyle='--')
axes[0, 0].set_xlabel('Gia tri du bao (fitted)')
axes[0, 0].set_ylabel('Phan du')
axes[0, 0].set_title('Phan du vs Gia tri du bao')

sm.qqplot(resid, line='45', fit=True, ax=axes[0, 1])
axes[0, 1].set_title('Q-Q plot phan du')

axes[1, 0].hist(resid, bins=10, edgecolor='white')
axes[1, 0].set_title('Histogram phan du')

axes[1, 1].stem(range(len(cooks_d)), cooks_d, markerfmt=',')
axes[1, 1].set_title("Khoang cach Cook (Cook's Distance)")
axes[1, 1].axhline(4 / len(df), color='red', linestyle='--', label='Nguong 4/n')
axes[1, 1].set_xlabel('Chi so quan sat')
axes[1, 1].legend()

plt.tight_layout()
plt.savefig(f'{FIG_DIR}/06_residual_diagnostics.png', dpi=150)
plt.show()


In [ ]:
shapiro_stat, shapiro_p = stats.shapiro(resid)
print(f"Shapiro-Wilk test phan du: W = {shapiro_stat:.4f}, p = {shapiro_p:.4f}")
print("-> Phan du", "co ve phan phoi chuan (p >= 0.05)" if shapiro_p >= 0.05 else "khong phan phoi chuan (p < 0.05)")

bp_stat, bp_p, _, _ = sm.stats.diagnostic.het_breuschpagan(resid, X)
print(f"\nBreusch-Pagan test (phuong sai dong nhat): LM stat = {bp_stat:.4f}, p = {bp_p:.4f}")
print("->", "Khong co bang chung phuong sai thay doi (p >= 0.05)" if bp_p >= 0.05 else "Co dau hieu phuong sai thay doi (p < 0.05)")


In [ ]:
df['Cooks_D'] = cooks_d
top_influence = df[['City', 'Walk', 'Bank', 'Talk', 'Heart', 'Cooks_D']].sort_values('Cooks_D', ascending=False).head(5)
top_influence


## 7. Hồi quy đơn biến với chỉ số nhịp sống tổng hợp

Ngoài mô hình 3 biến, ta chạy thêm hồi quy đơn giản `Heart ~ Pace_Index` — dễ diễn giải hơn và cho một
con số tổng hợp duy nhất để trả lời trực tiếp câu hỏi nghiên cứu.

In [ ]:
X_simple = sm.add_constant(df['Pace_Index'])
model_simple = sm.OLS(df['Heart'], X_simple).fit()
print(model_simple.summary())


## 8. Ước lượng hồi quy tuyến tính Bayes

Ở mục 6–7, mô hình được ước lượng theo trường phái **tần suất** (OLS): hệ số hồi quy là các hằng số chưa
biết được ước lượng bằng một điểm, và ý nghĩa thống kê đánh giá qua p-value. Mục này ước lượng **cùng một
mô hình** theo trường phái **Bayes**: các tham số `β` và `σ²` được coi là **biến ngẫu nhiên**, ta bắt đầu
từ một phân phối tiên nghiệm (prior), cập nhật bằng dữ liệu qua hàm hợp lý (likelihood) để thu được phân
phối **hậu nghiệm** (posterior) — nêu đầy đủ độ bất định của từng tham số, không chỉ một điểm ước lượng.

Công thức nền tảng (định lý Bayes):

```
p(β, σ² | y, X)  ∝  p(y | X, β, σ²) · p(β, σ²)
   hậu nghiệm       hàm hợp lý           prior
```

### 8.1. Thiết lập mô hình Bayes

**Hàm hợp lý (likelihood):** `y = Xβ + ε`, `ε ~ N(0, σ²·I)`, tức `y | X, β, σ² ~ N(Xβ, σ²·I)`, trong đó `X`
là ma trận thiết kế (cột hằng số + Walk + Bank + Talk), `y` là vector `Heart`, `n = 36`, `p = 4`.

**Prior liên hợp Normal-Inverse-Gamma (NIG):**

- `β | σ² ~ N(β₀, σ²·V₀)` — prior cho vector hệ số hồi quy.
- `σ² ~ Inverse-Gamma(a₀, b₀)` — prior cho phương sai sai số.

"Liên hợp" (conjugate) nghĩa là hậu nghiệm cũng thuộc họ NIG, nên có công thức dạng đóng.

**Chọn siêu tham số (weakly informative / prior mở):** `β₀ = 0`, `V₀ = 1000·I` (phương sai prior rất lớn,
gần như không áp đặt giá trị nào cho hệ số), `a₀ = b₀ = 0.001` (xấp xỉ prior Jeffreys cho `σ²`). Lý do:
không có thông tin tiên nghiệm đáng tin về hệ số hồi quy giữa nhịp sống và bệnh tim, nên để dữ liệu quyết
định hậu nghiệm. Hệ quả có chủ đích: kết quả Bayes sẽ gần sát OLS, cho phép **đối chiếu để kiểm tra tính
đúng đắn** của thuật toán.

### 8.2. Hậu nghiệm dạng đóng (closed-form)

Do tính liên hợp, hậu nghiệm là NIG với tham số:

```
Vn    = (V0⁻¹ + X'X)⁻¹
βn    = Vn (V0⁻¹ β0 + X'y)
an    = a0 + n/2
bn    = b0 + ½ ( y'y + β0' V0⁻¹ β0 − βn' Vn⁻¹ βn )
```

`βn` chính là trung bình hậu nghiệm của `β`. Khi prior rất mở (`V₀ → ∞`), `βn` tiến tới nghiệm OLS.

In [ ]:
def bayes_posterior_analytic(X, y, beta0=None, V0_scale=1000.0, a0=1e-3, b0=1e-3):
    """Hau nghiem dang dong cua hoi quy tuyen tinh Bayes voi prior lien hop Normal-Inverse-Gamma."""
    X = np.asarray(X, dtype=float)
    y = np.asarray(y, dtype=float)
    n, p = X.shape
    if beta0 is None:
        beta0 = np.zeros(p)
    V0_inv = np.linalg.inv(V0_scale * np.eye(p))
    Vn = np.linalg.inv(V0_inv + X.T @ X)
    beta_n = Vn @ (V0_inv @ beta0 + X.T @ y)
    an = a0 + n / 2
    bn = b0 + 0.5 * (y @ y + beta0 @ V0_inv @ beta0 - beta_n @ np.linalg.inv(Vn) @ beta_n)
    return {'Vn': Vn, 'beta_n': beta_n, 'an': an, 'bn': bn}

post_analytic = bayes_posterior_analytic(X.values, y.values)
compare = pd.DataFrame({'OLS (tan suat)': model.params.values,
                        'Trung binh hau nghiem (Bayes, closed-form)': post_analytic['beta_n']},
                       index=X.columns)
compare


**Nhận xét:** trung bình hậu nghiệm dạng đóng khớp gần như tuyệt đối với hệ số OLS — đúng như kỳ vọng với
prior mở, đồng thời xác nhận công thức được cài đặt chính xác. Điểm khác biệt của Bayes không nằm ở điểm
ước lượng mà ở **toàn bộ phân phối hậu nghiệm** (khoảng tin cậy Bayes, xác suất hậu nghiệm) như trình bày
tiếp theo.

### 8.3. Lấy mẫu hậu nghiệm bằng Gibbs sampling (MCMC)

Để minh hoạ **quá trình** ước lượng Bayes bằng mô phỏng (thay vì chỉ dùng công thức đóng), ta cài đặt
thuật toán **Gibbs sampling** — một dạng Markov Chain Monte Carlo (MCMC). Ý tưởng: thay vì lấy mẫu trực
tiếp từ hậu nghiệm chung `p(β, σ² | y)`, ta luân phiên lấy mẫu từ hai phân phối điều kiện đầy đủ (full
conditional), mỗi phân phối đều có dạng đã biết:

```
σ² | β, y   ~  Inverse-Gamma( a0 + (n+p)/2 ,  b0 + ½ [ (y−Xβ)'(y−Xβ) + (β−β0)' V0⁻¹ (β−β0) ] )
β  | σ², y  ~  N( βn ,  σ² · Vn )
```

Lặp lại nhiều lần, chuỗi (β, σ²) hội tụ về phân phối hậu nghiệm. Ta chạy **4 chain độc lập** với điểm khởi
tạo khác nhau, mỗi chain **6000 vòng lặp**, **bỏ 1000 vòng đầu (burn-in)** vì chuỗi chưa ổn định.

In [ ]:
def gibbs_bayes_linear_regression(X, y, beta0=None, V0_scale=1000.0, a0=1e-3, b0=1e-3,
                                   n_iter=6000, burn_in=1000, n_chains=4, seed=42):
    X = np.asarray(X, dtype=float)
    y = np.asarray(y, dtype=float)
    n, p = X.shape
    if beta0 is None:
        beta0 = np.zeros(p)
    V0_inv = np.linalg.inv(V0_scale * np.eye(p))
    Vn = np.linalg.inv(V0_inv + X.T @ X)
    beta_n = Vn @ (V0_inv @ beta0 + X.T @ y)

    rng = np.random.default_rng(seed)
    chains_beta = np.zeros((n_chains, n_iter, p))
    chains_sigma2 = np.zeros((n_chains, n_iter))
    for c in range(n_chains):
        beta_curr = beta_n + rng.normal(scale=8.0, size=p)   # diem khoi tao khac nhau giua cac chain
        for t in range(n_iter):
            resid = y - X @ beta_curr
            diff = beta_curr - beta0
            a_n = a0 + (n + p) / 2
            b_n = b0 + 0.5 * (resid @ resid + diff @ V0_inv @ diff)
            sigma2_curr = 1.0 / rng.gamma(shape=a_n, scale=1.0 / b_n)     # sigma^2 | beta, y
            beta_curr = rng.multivariate_normal(beta_n, sigma2_curr * Vn)  # beta | sigma^2, y
            chains_beta[c, t] = beta_curr
            chains_sigma2[c, t] = sigma2_curr
    return {'chains_beta': chains_beta, 'chains_sigma2': chains_sigma2, 'burn_in': burn_in}

gibbs_main = gibbs_bayes_linear_regression(X.values, y.values)
print('Da chay xong Gibbs sampling: 4 chain x 6000 vong lap (bo 1000 vong burn-in) cho Heart ~ Walk + Bank + Talk.')


### 8.4. Chẩn đoán hội tụ (trace plot và Gelman-Rubin R-hat)

Kết quả MCMC chỉ đáng tin nếu các chain đã hội tụ. Có hai kiểm tra chuẩn:

- **Trace plot:** vẽ giá trị tham số theo vòng lặp cho từng chain. Nếu hội tụ, các chain dao động quanh
  cùng một mức và **chồng lấn lên nhau** (không trôi dạt, không tách rời).
- **Gelman-Rubin R-hat:** so sánh phương sai *giữa* các chain với phương sai *trong* từng chain. R-hat ≈ 1
  (thường dùng ngưỡng < 1.05, chặt hơn < 1.01) cho thấy các chain cùng mô tả một phân phối.

In [ ]:
def gelman_rubin(chains_2d):
    m, n = chains_2d.shape
    chain_means = chains_2d.mean(axis=1)
    chain_vars = chains_2d.var(axis=1, ddof=1)
    W = chain_vars.mean()
    B = n * chain_means.var(ddof=1)
    var_hat = (1 - 1 / n) * W + B / n
    return np.sqrt(var_hat / W) if W > 0 else np.nan

param_names = list(X.columns)
burn_in = gibbs_main['burn_in']
post_beta = gibbs_main['chains_beta'][:, burn_in:, :]
post_sigma2 = gibbs_main['chains_sigma2'][:, burn_in:]

fig, axes = plt.subplots(len(param_names) + 1, 1, figsize=(10, 2.2 * (len(param_names) + 1)))
for j, name in enumerate(param_names):
    for c in range(post_beta.shape[0]):
        axes[j].plot(post_beta[c, :, j], alpha=0.7, linewidth=0.6)
    axes[j].set_title(f'Trace plot: beta ({name})')
for c in range(post_sigma2.shape[0]):
    axes[-1].plot(post_sigma2[c], alpha=0.7, linewidth=0.6)
axes[-1].set_title('Trace plot: sigma^2')
plt.tight_layout()
plt.savefig(f'{FIG_DIR}/07_bayes_trace_plots.png', dpi=150)
plt.show()

print(f"{'Tham so':12s} {'R-hat (Gelman-Rubin)':>22s}")
print('-' * 36)
for j, name in enumerate(param_names):
    print(f"{name:12s} {gelman_rubin(post_beta[:, :, j]):22.4f}")
print(f"{'sigma^2':12s} {gelman_rubin(post_sigma2):22.4f}")


**Nhận xét:** nếu tất cả R-hat xấp xỉ 1.000 và trace plot của 4 chain chồng lấn nhau, các chain đã hội tụ về
cùng một phân phối hậu nghiệm và mẫu thu được (4 × 5000 = 20.000 mẫu sau burn-in) đáng tin cậy để tổng hợp.

### 8.5. Tổng hợp hậu nghiệm và đối chiếu với OLS

Từ các mẫu hậu nghiệm, ta tính với mỗi hệ số:

- **Trung bình / độ lệch chuẩn hậu nghiệm:** điểm ước lượng và độ bất định.
- **Khoảng tin cậy Bayes 95% (credible interval):** khoảng chứa hệ số với xác suất 95% *theo hậu nghiệm* —
  diễn giải trực tiếp hơn khoảng tin cậy tần suất.
- **`P(β > 0 | dữ liệu)`:** xác suất hậu nghiệm để hệ số dương. Đây là câu trả lời trực tiếp cho câu hỏi "tác
  động có cùng chiều với giả thuyết không?", không cần ngưỡng p-value tuỳ ý.

In [ ]:
pooled_beta = post_beta.reshape(-1, post_beta.shape[-1])
rows = []
for j, name in enumerate(param_names):
    s = pooled_beta[:, j]
    rows.append({
        'Bien': name,
        'OLS_coef': model.params[name],
        'OLS_se': model.bse[name],
        'Bayes_mean': s.mean(),
        'Bayes_sd': s.std(ddof=1),
        'CI_2.5%': np.percentile(s, 2.5),
        'CI_97.5%': np.percentile(s, 97.5),
        'P(beta>0|data)': (s > 0).mean(),
    })
bayes_summary = pd.DataFrame(rows).set_index('Bien')
bayes_summary


In [ ]:
fig, axes = plt.subplots(1, len(param_names), figsize=(4.2 * len(param_names), 4))
for j, (ax, name) in enumerate(zip(axes, param_names)):
    s = pooled_beta[:, j]
    ax.hist(s, bins=50, color='#4C72B0', alpha=0.8, density=True)
    lo, hi = np.percentile(s, [2.5, 97.5])
    ax.axvline(lo, color='red', linestyle='--', linewidth=1)
    ax.axvline(hi, color='red', linestyle='--', linewidth=1)
    ax.axvline(0, color='black', linewidth=1)
    ax.set_title(f'Hau nghiem beta ({name})')
plt.tight_layout()
plt.savefig(f'{FIG_DIR}/08_bayes_posterior_hist.png', dpi=150)
plt.show()


**Cách đọc biểu đồ:** đường đứt đỏ là hai đầu khoảng tin cậy Bayes 95%; đường đen là mốc 0. Hệ số có bằng
chứng hậu nghiệm rõ ràng khi phần lớn khối lượng phân phối nằm về một phía của mốc 0 (khoảng tin cậy không
chứa 0, `P(β>0|data)` gần 1 hoặc gần 0).

### 8.6. Ước lượng Bayes cho mô hình đơn biến với `Pace_Index`

Áp dụng cùng thuật toán cho mô hình `Heart ~ Pace_Index` (mục 7) để có đối chiếu Bayes cho chỉ số tổng hợp.

In [ ]:
gibbs_simple = gibbs_bayes_linear_regression(X_simple.values, df['Heart'].values)
post_simple = gibbs_simple['chains_beta'][:, gibbs_simple['burn_in']:, :].reshape(-1, 2)
rows = []
for j, name in enumerate(['const', 'Pace_Index']):
    s = post_simple[:, j]
    rows.append({'Bien': name, 'OLS_coef': model_simple.params[name], 'Bayes_mean': s.mean(),
                 'Bayes_sd': s.std(ddof=1), 'CI_2.5%': np.percentile(s, 2.5),
                 'CI_97.5%': np.percentile(s, 97.5), 'P(beta>0|data)': (s > 0).mean()})
simple_summary = pd.DataFrame(rows).set_index('Bien')
simple_summary


## 9. Đánh giá kết quả và kết luận

In [ ]:
r_walk, p_walk = stats.pearsonr(df['Walk'], df['Heart'])
r_bank, p_bank = stats.pearsonr(df['Bank'], df['Heart'])
r_talk, p_talk = stats.pearsonr(df['Talk'], df['Heart'])
r_pace, p_pace = stats.pearsonr(df['Pace_Index'], df['Heart'])

def nhan_xet(name, r, p, expect_positive=True):
    sig = p < 0.05
    khop = (r > 0) == expect_positive
    if sig and khop:
        return f'{name}: UNG HO gia thuyet, co y nghia thong ke (r={r:+.3f}, p={p:.3f})'
    if sig and not khop:
        return f'{name}: NGUOC voi gia thuyet, co y nghia thong ke (r={r:+.3f}, p={p:.3f})'
    if khop:
        return f'{name}: dung chieu gia thuyet nhung KHONG co y nghia thong ke (r={r:+.3f}, p={p:.3f})'
    return f'{name}: nguoc chieu gia thuyet va KHONG co y nghia thong ke (r={r:+.3f}, p={p:.3f})'

print('=' * 84)
print('TOM TAT KET QUA (TAN SUAT + BAYES)')
print('=' * 84)
print('--- Tan suat (Pearson) so voi ky vong type-A: Walk cao, Talk cao, Bank THAP => Heart cao ---')
print(' -', nhan_xet('Walk', r_walk, p_walk, True))
print(' -', nhan_xet('Talk', r_talk, p_talk, True))
print(' -', nhan_xet('Bank', r_bank, p_bank, False))
print(f"\n--- Hoi quy boi OLS: R2 = {model.rsquared:.4f}, Adj R2 = {model.rsquared_adj:.4f}, F = {model.fvalue:.3f}, p = {model.f_pvalue:.4g}")
print('\n--- Bayes (hau nghiem Gibbs sampling) ---')
for name in param_names:
    r = bayes_summary.loc[name]
    print(f"  {name:8s} mean = {r['Bayes_mean']:+.4f}, 95% CI = [{r['CI_2.5%']:+.4f}, {r['CI_97.5%']:+.4f}], P(beta>0|data) = {r['P(beta>0|data)']:.3f}")
print(f"\n--- Pace_Index: r = {r_pace:+.3f}, p = {p_pace:.4f}; OLS R2 = {model_simple.rsquared:.4f}; "
      f"Bayes P(beta>0|data) = {simple_summary.loc['Pace_Index', 'P(beta>0|data)']:.3f}")


**Biện luận trả lời câu hỏi nghiên cứu:**

- **Hai trường phái cho kết quả thống nhất:** hệ số OLS và trung bình hậu nghiệm Bayes gần như trùng khớp
  (do prior mở); R-hat ≈ 1 xác nhận thuật toán Gibbs hội tụ. Khác biệt nằm ở cách diễn giải: p-value cho
  biết dữ liệu "bất thường" đến mức nào nếu giả thuyết không đúng, còn `P(β>0|dữ liệu)` cho biết trực tiếp
  mức độ tin vào chiều tác động sau khi thấy dữ liệu.
- **`Walk`** là bằng chứng mạnh và nhất quán nhất: hệ số dương, có ý nghĩa tần suất và xác suất hậu nghiệm
  dương rất cao — ủng hộ giả thuyết type-A (đi bộ nhanh ⇒ tỷ lệ bệnh tim cao hơn).
- **`Bank`** có hệ số dương với xác suất hậu nghiệm cao, nhưng **ngược chiều** dự đoán ("Bank thấp = gấp
  gáp = bệnh tim cao"): thành phố xử lý giao dịch chậm hơn mới có bệnh tim cao hơn. Đây là phát hiện thực
  nghiệm, không phải lỗi tính toán.
- **`Talk`** có hệ số âm, khoảng tin cậy Bayes chứa 0 và `P(β>0|dữ liệu)` thấp — không có bằng chứng rõ
  ràng khi đã kiểm soát Walk và Bank.
- **`Pace_Index`** (giả định Bank thấp = gấp gáp) không phải thước đo phù hợp với dữ liệu này: thành phần
  Bank triệt tiêu tín hiệu của Walk; cả OLS lẫn Bayes đều cho khoảng tin cậy rất rộng, chứa 0.

**Kết luận:** dữ liệu **ủng hộ một phần, không dứt khoát** niềm tin rằng cảm giác gấp gáp về thời gian đi
kèm nguy cơ bệnh tim cao hơn; bằng chứng mạnh nhất đến từ tốc độ đi bộ (`Walk`), được xác nhận độc lập bởi
cả kiểm định tần suất lẫn xác suất hậu nghiệm Bayes.

**Các giới hạn cần lưu ý:**

1. **Cỡ mẫu nhỏ** (36 thành phố) làm tăng độ bất định ở cả hai trường phái (khoảng tin cậy rộng).
2. **Dữ liệu cấp độ tổng hợp (ecological data):** chỉ số là đại diện của thành phố, không đo trên từng cá
   nhân — có nguy cơ *ecological fallacy*, không suy trực tiếp nhân quả ở cấp cá nhân.
3. **Tương quan không suy ra nhân quả:** các yếu tố gây nhiễu (khí hậu, chế độ ăn, thu nhập, hút thuốc,
   chăm sóc y tế, cơ cấu tuổi...) không được kiểm soát.
4. **Prior mở được chọn có chủ đích** để đối chiếu với OLS; nếu có thông tin tiên nghiệm thực tế, kết quả
   Bayes có thể khác và cần phân tích độ nhạy (sensitivity analysis) với các prior khác.
5. **R² của mô hình OLS** cho biết phần lớn biến thiên của tỷ lệ bệnh tim vẫn do các yếu tố khác quyết
   định.